# 07 — Permit Ground Truth Audit

**Learning goal:** Understand the quality of permit-derived training labels.
Building permits tell us *when* reconstruction happened, which we can use to
label satellite time-series. But permits are noisy — not every permit means
what it says.

**Key question:** What is the actual quality of the permit labels?

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
TABULAR_DIR = DATA_DIR / "tabular"

print(f"RAW_DIR: {RAW_DIR.resolve()}")
print(f"TABULAR_DIR: {TABULAR_DIR.resolve()}")

## Load Parcels, Permits, and FEMA Damage Data

We need three datasets:
1. **Parcels** — polygon geometries with parcel IDs
2. **FEMA damage** — official damage assessment (destroyed/major/minor/none)
3. **Building permits** — reconstruction timeline from Boulder County

In [ ]:
# --- Load parcels ---
parcels = None
parcel_candidates = [
    RAW_DIR / "parcels.geojson",
    RAW_DIR / "boulder_parcels.geojson",
    TABULAR_DIR / "parcels.geojson",
]
for p in parcel_candidates:
    if p.exists():
        parcels = gpd.read_file(p)
        print(f"Parcels: {len(parcels)} features from {p.name}")
        break
if parcels is None:
    print("No parcel file found. Expected one of:")
    for p in parcel_candidates:
        print(f"  {p}")

# --- Load FEMA damage ---
fema = None
fema_candidates = [
    RAW_DIR / "fema_damage.csv",
    TABULAR_DIR / "fema_damage.csv",
    RAW_DIR / "damage_assessment.csv",
    TABULAR_DIR / "damage_assessment.csv",
]
for p in fema_candidates:
    if p.exists():
        fema = pd.read_csv(p)
        print(f"FEMA damage: {len(fema)} rows from {p.name}")
        break
if fema is None:
    print("No FEMA damage file found.")

# --- Load permits ---
permits = None
permit_candidates = [
    RAW_DIR / "permits.csv",
    TABULAR_DIR / "permits.csv",
    RAW_DIR / "building_permits.csv",
    TABULAR_DIR / "building_permits.csv",
]
for p in permit_candidates:
    if p.exists():
        permits = pd.read_csv(p)
        print(f"Permits: {len(permits)} rows from {p.name}")
        break
if permits is None:
    print("No permits file found.")

# --- Check parcel_id format consistency ---
parcel_id_col = None
if parcels is not None:
    for col in ["parcel_id", "PARCEL_ID", "parcelid", "APN", "PIN"]:
        if col in parcels.columns:
            parcel_id_col = col
            break
    if parcel_id_col:
        sample_ids = parcels[parcel_id_col].head(5).tolist()
        print(f"\nParcel ID column: '{parcel_id_col}'")
        print(f"Sample IDs: {sample_ids}")
        print(f"ID dtype: {parcels[parcel_id_col].dtype}")

        if fema is not None and parcel_id_col in fema.columns:
            fema_ids = fema[parcel_id_col].head(5).tolist()
            print(f"FEMA ID samples: {fema_ids}")
            print(f"FEMA ID dtype: {fema[parcel_id_col].dtype}")

            # Check overlap
            overlap = set(parcels[parcel_id_col]) & set(fema[parcel_id_col])
            print(f"Overlapping IDs: {len(overlap)}")
    else:
        print("Could not identify parcel ID column.")
        print(f"Available columns: {list(parcels.columns)}")

## Join on parcel_id — No Geocoding Needed

Because both datasets use parcel IDs, we can do a simple tabular join
rather than a slow spatial join.

In [ ]:
merged = None
damage_col = None

if parcels is not None and fema is not None and parcel_id_col is not None:
    # Ensure consistent types before join
    parcels[parcel_id_col] = parcels[parcel_id_col].astype(str).str.strip()
    fema[parcel_id_col] = fema[parcel_id_col].astype(str).str.strip()

    merged = parcels.merge(fema, on=parcel_id_col, how="inner")
    print(f"Joined parcels + FEMA: {len(merged)} rows")
    print(f"  Parcels total:     {len(parcels)}")
    print(f"  FEMA total:        {len(fema)}")
    print(f"  Inner join result: {len(merged)}")

    # Show available columns
    print(f"\nMerged columns: {list(merged.columns)}")

    # Find damage class column
    for col in ["damage_class", "damage_category", "DAMAGE", "damage", "damage_level"]:
        if col in merged.columns:
            damage_col = col
            break

    if damage_col:
        print(f"\nDamage distribution ({damage_col}):")
        print(merged[damage_col].value_counts())

    print("\nFirst 5 rows:")
    display(merged.head())
else:
    print("Cannot perform join — missing parcels or FEMA data.")
    print("\nTo proceed, you need:")
    print("  1. Parcel geometries (GeoJSON with parcel_id)")
    print("  2. FEMA damage assessment (CSV with parcel_id and damage class)")

## REBUILD_PERMIT_TYPES Classification

Building permits reveal the reconstruction stage. We map permit descriptions
to a standardized `implied_stage`:

| Permit keyword | implied_stage |
|----------------|---------------|
| demolition | cleared_lot |
| foundation | foundation_framing |
| framing / structural | structure_substantial |
| certificate of occupancy (COO) | rebuild_complete |
| new dwelling / new SFR | rebuild_start |

In [ ]:
REBUILD_PERMIT_TYPES = [
    "demolition",
    "foundation",
    "framing",
    "structural",
    "new dwelling",
    "new sfr",
    "new single family",
    "certificate of occupancy",
    "coo",
    "final inspection",
    "rebuild",
]


def classify_permit_stage(description):
    """Map a permit description to an implied reconstruction stage."""
    if pd.isna(description):
        return "unknown"
    desc_lower = str(description).lower()

    if any(kw in desc_lower for kw in ["demolition", "demo", "tear down"]):
        return "cleared_lot"
    elif any(kw in desc_lower for kw in ["foundation", "footing", "slab"]):
        return "foundation_framing"
    elif any(kw in desc_lower for kw in ["framing", "structural", "rough"]):
        return "structure_substantial"
    elif any(kw in desc_lower for kw in ["certificate of occupancy", "coo", "final", "c of o"]):
        return "rebuild_complete"
    elif any(kw in desc_lower for kw in ["new dwelling", "new sfr", "new single", "new residence", "rebuild"]):
        return "rebuild_start"
    else:
        return "other"


rebuild_permits = None

if permits is not None:
    # Find description column
    desc_col = None
    for col in ["description", "DESCRIPTION", "permit_type", "PERMIT_TYPE",
                "work_description", "WORK_DESC", "type"]:
        if col in permits.columns:
            desc_col = col
            break

    if desc_col:
        print(f"Using description column: '{desc_col}'")
        print("Sample descriptions:")
        print(permits[desc_col].head(10).to_string())

        # Classify
        permits["implied_stage"] = permits[desc_col].apply(classify_permit_stage)

        print("\nImplied stage distribution:")
        print(permits["implied_stage"].value_counts())

        # Filter to rebuild-related permits only
        rebuild_permits = permits[permits["implied_stage"] != "other"].copy()
        print(f"\nRebuild-related permits: {len(rebuild_permits)} / {len(permits)}")
    else:
        print(f"Could not find description column. Available: {list(permits.columns)}")
else:
    print("No permits CSV found.")
    print("\nTo obtain permits data, download from Boulder County open data portal:")
    print("  https://opendata-bouldercounty.hub.arcgis.com/")
    print("  Search for 'building permits' and filter to Marshall Fire area (2022-2024).")
    print("  Save as CSV to data/raw/permits.csv or data/tabular/permits.csv")

## Label Confidence Assessment

Not all permit-derived labels are equally reliable. We assign a confidence score:

- **high** — Certificate of occupancy or framing inspection (unambiguous physical state)
- **medium** — Foundation permit or demolition (clear intent but state less certain)
- **low** — Generic "new dwelling" or ambiguous descriptions

In [ ]:
def label_confidence(implied_stage):
    """Assign confidence level based on implied reconstruction stage.

    High: physical state is unambiguous from the permit type.
    Medium: clear intent but actual state less certain.
    Low: generic or ambiguous permit descriptions.
    """
    if implied_stage in ["rebuild_complete", "structure_substantial"]:
        return "high"
    elif implied_stage in ["foundation_framing", "cleared_lot"]:
        return "medium"
    elif implied_stage in ["rebuild_start"]:
        return "low"
    else:
        return "low"


if rebuild_permits is not None:
    rebuild_permits["label_confidence"] = rebuild_permits["implied_stage"].apply(label_confidence)

    print("Label confidence distribution:")
    conf_counts = rebuild_permits["label_confidence"].value_counts()
    print(conf_counts)
    print(f"\nTotal rebuild permits: {len(rebuild_permits)}")
    print(f"High confidence: {conf_counts.get('high', 0)} ({100*conf_counts.get('high', 0)/len(rebuild_permits):.1f}%)")
    print(f"Medium confidence: {conf_counts.get('medium', 0)} ({100*conf_counts.get('medium', 0)/len(rebuild_permits):.1f}%)")
    print(f"Low confidence: {conf_counts.get('low', 0)} ({100*conf_counts.get('low', 0)/len(rebuild_permits):.1f}%)")

    # Cross-tab: stage x confidence
    print("\nStage x Confidence cross-tab:")
    print(pd.crosstab(rebuild_permits["implied_stage"], rebuild_permits["label_confidence"]))
else:
    print("No rebuild permits available for confidence assessment.")
    print("Confidence scheme:")
    print("  high   - COO, framing inspection")
    print("  medium - foundation, demolition")
    print("  low    - generic new dwelling")

## Visualize Confidence on Map

Plot parcels colored by label confidence. Green = high confidence labels we can
trust for training. Orange = medium. Red = low confidence, use with caution.

In [ ]:
CONFIDENCE_COLORS = {
    "high": "#2ecc71",    # green
    "medium": "#f39c12",  # orange
    "low": "#e74c3c",     # red
}

if rebuild_permits is not None and parcels is not None and parcel_id_col is not None:
    # Find parcel ID in permits
    permit_id_col = None
    for col in [parcel_id_col, "parcel_id", "PARCEL_ID", "parcelid", "APN"]:
        if col in rebuild_permits.columns:
            permit_id_col = col
            break

    if permit_id_col:
        # Take the highest-confidence label per parcel (if multiple permits)
        conf_order = {"high": 0, "medium": 1, "low": 2}
        rebuild_permits["_conf_rank"] = rebuild_permits["label_confidence"].map(conf_order)
        best_per_parcel = (
            rebuild_permits
            .sort_values("_conf_rank")
            .drop_duplicates(subset=[permit_id_col], keep="first")
        )
        best_per_parcel[permit_id_col] = best_per_parcel[permit_id_col].astype(str).str.strip()

        # Join to parcels
        parcels_copy = parcels.copy()
        parcels_copy[parcel_id_col] = parcels_copy[parcel_id_col].astype(str).str.strip()
        parcels_with_conf = parcels_copy.merge(
            best_per_parcel[[permit_id_col, "implied_stage", "label_confidence"]],
            on=parcel_id_col,
            how="inner",
        )
        print(f"Parcels with permit labels: {len(parcels_with_conf)}")

        # Plot
        fig, ax = plt.subplots(figsize=(12, 10))

        # Plot all parcels as light gray background
        parcels.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#cccccc", linewidth=0.3)

        # Overlay permit-labeled parcels colored by confidence
        for conf_level, color in CONFIDENCE_COLORS.items():
            subset = parcels_with_conf[parcels_with_conf["label_confidence"] == conf_level]
            if len(subset) > 0:
                subset.plot(ax=ax, facecolor=color, edgecolor="black",
                           linewidth=0.5, alpha=0.7)

        # Legend
        legend_patches = [
            mpatches.Patch(facecolor=CONFIDENCE_COLORS["high"], label=f"High ({len(parcels_with_conf[parcels_with_conf['label_confidence']=='high'])})"),
            mpatches.Patch(facecolor=CONFIDENCE_COLORS["medium"], label=f"Medium ({len(parcels_with_conf[parcels_with_conf['label_confidence']=='medium'])})"),
            mpatches.Patch(facecolor=CONFIDENCE_COLORS["low"], label=f"Low ({len(parcels_with_conf[parcels_with_conf['label_confidence']=='low'])})"),
            mpatches.Patch(facecolor="#f0f0f0", edgecolor="#cccccc", label="No permit"),
        ]
        ax.legend(handles=legend_patches, loc="upper right", title="Label Confidence")

        ax.set_title("Marshall Fire — Permit Label Confidence by Parcel", fontsize=14)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        plt.tight_layout()
        plt.show()
    else:
        print("Could not find parcel ID column in permits.")
        print(f"Permits columns: {list(rebuild_permits.columns)}")
else:
    print("Cannot create map — missing parcels or permit data.")
    print("\nExpected data layout:")
    print("  data/raw/parcels.geojson — parcel polygons with parcel_id")
    print("  data/raw/permits.csv — permit records with parcel_id and description")
    print("\nWhen data is available, the map will show:")
    print("  Green  = high confidence labels (COO, framing inspections)")
    print("  Orange = medium confidence (foundation, demolition)")
    print("  Red    = low confidence (generic permits)")
    print("  Gray   = no permit data for that parcel")